# 01 – Data inspection and initial quality checks

**Purpose.** Understand the structure of the two source files and record data-quality issues *before* any exploration or modelling.

**Scope of this notebook.**
- The source CSVs are only read, never written.
- No cleaning is applied. Issues are recorded so we can choose treatments later.
- Missing hourly timestamps stay missing. Nothing is filled in.

**What `cnt` means.** It is the number of rentals *started* in that hour (or day). That is not the same as the number of bikes needed. This notebook only checks the counts and does not interpret them.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

# Paths are relative to this notebook: notebooks/ -> data/
DATA_DIR = Path("..") / "data"
HOUR_PATH = DATA_DIR / "hour.csv"
DAY_PATH = DATA_DIR / "day.csv"

# Collects one row per check so we can print a summary table at the end.
results = []
def record(check, result, status):
    results.append({"check": check, "result": result, "status": status})

## 1. Load the files

Everything below depends on reading the files correctly. We read them as-is with default parsing. Dates stay as text here and are parsed explicitly in section 3.

In [2]:
hour_raw = pd.read_csv(HOUR_PATH)
day_raw = pd.read_csv(DAY_PATH)

# Work on copies so the as-loaded frames stay available for comparison.
hour = hour_raw.copy()
day = day_raw.copy()
print("hour.csv:", HOUR_PATH)
print("day.csv: ", DAY_PATH)

hour.csv: ..\data\hour.csv
day.csv:  ..\data\day.csv


## 2. Dimensions, columns, data types, sample

This confirms the files match the README's description: expected columns, numeric types where we expect numbers, and a plausible row count.

In [3]:
for name, df in [("hour", hour), ("day", day)]:
    print(f"--- {name}: {df.shape[0]:,} rows x {df.shape[1]} columns")
    print(df.dtypes.to_string())
    print()

record("hour.csv shape", f"{hour.shape[0]:,} rows x {hour.shape[1]} cols", "info")
record("day.csv shape", f"{day.shape[0]:,} rows x {day.shape[1]} cols", "info")

# Columns that appear in only one file (README says only `hr` should differ)
print("Columns only in hour.csv:", sorted(set(hour.columns) - set(day.columns)))
print("Columns only in day.csv: ", sorted(set(day.columns) - set(hour.columns)))

--- hour: 17,379 rows x 17 columns
instant         int64
dteday            str
season          int64
yr              int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
casual          int64
registered      int64
cnt             int64

--- day: 731 rows x 16 columns
instant         int64
dteday            str
season          int64
yr              int64
mnth            int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
casual          int64
registered      int64
cnt             int64

Columns only in hour.csv: ['hr']
Columns only in day.csv:  []


In [4]:
hour.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,01-01-2011,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,01-01-2011,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,01-01-2011,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,01-01-2011,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,01-01-2011,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


In [5]:
day.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,01-01-2011,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,02-01-2011,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,03-01-2011,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,04-01-2011,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,05-01-2011,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


## 3. Parse dates and build the hourly timestamp

Every time-based check (gaps, duplicates, daily totals) needs a real datetime. In the raw files the date looks like `01-01-2011`, which could be read as day-month or month-day.

We parse with an explicit `%d-%m-%Y` format and `errors="raise"`. If the format were wrong, parsing would fail on any day above 12. We also confirm that `day.csv` dates are consecutive in `instant` order and that the parsed month and year match the `mnth` and `yr` columns.

In [6]:
DATE_FORMAT = "%d-%m-%Y"
hour["date"] = pd.to_datetime(hour["dteday"], format=DATE_FORMAT, errors="raise")
day["date"] = pd.to_datetime(day["dteday"], format=DATE_FORMAT, errors="raise")

# Hourly timestamp = calendar date + hour of day
hour["timestamp"] = hour["date"] + pd.to_timedelta(hour["hr"], unit="h")

print("hour date range:", hour["date"].min().date(), "to", hour["date"].max().date())
print("day  date range:", day["date"].min().date(), "to", day["date"].max().date())
print("hr values:", sorted(hour["hr"].unique().tolist()))

# Sanity checks on the chosen date format
day_steps = day.sort_values("instant")["date"].diff().dropna().unique()
print("Gaps between consecutive day.csv dates:", day_steps)

mnth_match = ((hour["date"].dt.month == hour["mnth"]).all()
              and (day["date"].dt.month == day["mnth"]).all())
yr_match = ((hour["date"].dt.year - 2011 == hour["yr"]).all()
            and (day["date"].dt.year - 2011 == day["yr"]).all())
print("Parsed month matches `mnth`:", mnth_match)
print("Parsed year matches `yr` (0=2011, 1=2012):", yr_match)

record("Date parsing (%d-%m-%Y)",
       f"all parsed; {day['date'].min().date()} to {day['date'].max().date()}; "
       f"month match={mnth_match}, year match={yr_match}",
       "pass" if (mnth_match and yr_match) else "issue")

hour date range: 2011-01-01 to 2012-12-31
day  date range: 2011-01-01 to 2012-12-31
hr values: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
Gaps between consecutive day.csv dates: <TimedeltaArray>
['1 days']
Length: 1, dtype: timedelta64[us]
Parsed month matches `mnth`: True
Parsed year matches `yr` (0=2011, 1=2012): True


In [7]:
hour[["instant", "dteday", "hr", "timestamp"]].head()

,instant,dteday,hr,timestamp
0,1,01-01-2011,0,2011-01-01 00:00:00
1,2,01-01-2011,1,2011-01-01 01:00:00
2,3,01-01-2011,2,2011-01-01 02:00:00
3,4,01-01-2011,3,2011-01-01 03:00:00
4,5,01-01-2011,4,2011-01-01 04:00:00


## 4. Blank values and duplicates

Blank values would bias averages and break models without warning. Duplicate rows or timestamps would count some hours twice.

We check:
- full-row duplicates
- duplicates ignoring the `instant` row index, which is unique by construction and could hide a repeated record
- duplicate hourly timestamps and duplicate daily dates

In [8]:
for name, df in [("hour", hour_raw), ("day", day_raw)]:
    blanks = df.isna().sum()
    print(f"{name}: total blank cells = {int(blanks.sum())}")
    if blanks.sum() > 0:
        print(blanks[blanks > 0])

hour_blank = int(hour_raw.isna().sum().sum())
day_blank = int(day_raw.isna().sum().sum())
record("Blank values", f"hour={hour_blank}, day={day_blank}",
       "pass" if hour_blank == day_blank == 0 else "issue")

hour: total blank cells = 0
day: total blank cells = 0


In [9]:
hour_dup_rows = int(hour_raw.duplicated().sum())
day_dup_rows = int(day_raw.duplicated().sum())
hour_dup_no_id = int(hour_raw.drop(columns="instant").duplicated().sum())
day_dup_no_id = int(day_raw.drop(columns="instant").duplicated().sum())
hour_dup_ts = int(hour["timestamp"].duplicated().sum())
day_dup_dates = int(day["date"].duplicated().sum())
hour_dup_instant = int(hour["instant"].duplicated().sum())
day_dup_instant = int(day["instant"].duplicated().sum())

print("Full-row duplicates:          hour =", hour_dup_rows, " day =", day_dup_rows)
print("Duplicates ignoring instant:  hour =", hour_dup_no_id, " day =", day_dup_no_id)
print("Duplicate `instant` values:   hour =", hour_dup_instant, " day =", day_dup_instant)
print("Duplicate hourly timestamps: ", hour_dup_ts)
print("Duplicate daily dates:       ", day_dup_dates)

record("Duplicate rows (incl. ignoring instant)",
       f"hour={hour_dup_rows}/{hour_dup_no_id}, day={day_dup_rows}/{day_dup_no_id}",
       "pass" if hour_dup_rows + day_dup_rows + hour_dup_no_id + day_dup_no_id == 0 else "issue")
record("Duplicate hourly timestamps", str(hour_dup_ts), "pass" if hour_dup_ts == 0 else "issue")
record("Duplicate daily dates", str(day_dup_dates), "pass" if day_dup_dates == 0 else "issue")

Full-row duplicates:          hour = 0  day = 0
Duplicates ignoring instant:  hour = 0  day = 0
Duplicate `instant` values:   hour = 0  day = 0
Duplicate hourly timestamps:  0
Duplicate daily dates:        0


## 5. Rental count validity

Counts of rentals must be non-negative whole numbers. The README defines `cnt` as `casual + registered`, so if that identity fails, the target variable itself is unreliable.

In [10]:
COUNT_COLS = ["casual", "registered", "cnt"]

for name, df in [("hour", hour), ("day", day)]:
    print(f"--- {name}")
    for col in COUNT_COLS:
        s = df[col]
        n_neg = int((s < 0).sum())
        n_nonint = int((s % 1 != 0).sum())
        print(f"{col:>11}: dtype={s.dtype}, min={s.min()}, max={s.max()}, "
              f"negative={n_neg}, non-integer={n_nonint}")

neg_total = sum(int((df[c] < 0).sum()) for df in (hour, day) for c in COUNT_COLS)
nonint_total = sum(int((df[c] % 1 != 0).sum()) for df in (hour, day) for c in COUNT_COLS)
record("Negative counts (both files)", str(neg_total), "pass" if neg_total == 0 else "issue")
record("Non-integer counts (both files)", str(nonint_total), "pass" if nonint_total == 0 else "issue")

--- hour
     casual: dtype=int64, min=0, max=367, negative=0, non-integer=0
 registered: dtype=int64, min=0, max=886, negative=0, non-integer=0
        cnt: dtype=int64, min=1, max=977, negative=0, non-integer=0
--- day
     casual: dtype=int64, min=2, max=3410, negative=0, non-integer=0
 registered: dtype=int64, min=20, max=6946, negative=0, non-integer=0
        cnt: dtype=int64, min=22, max=8714, negative=0, non-integer=0


In [11]:
hour_sum_mismatch = int((hour["cnt"] != hour["casual"] + hour["registered"]).sum())
day_sum_mismatch = int((day["cnt"] != day["casual"] + day["registered"]).sum())
print("Rows where cnt != casual + registered:  hour =", hour_sum_mismatch, " day =", day_sum_mismatch)

record("cnt = casual + registered", f"mismatches: hour={hour_sum_mismatch}, day={day_sum_mismatch}",
       "pass" if hour_sum_mismatch == day_sum_mismatch == 0 else "issue")

Rows where cnt != casual + registered:  hour = 0  day = 0


In [12]:
# Hours with zero rentals: we report these so that "absent hour" and "hour with 0 rentals"
# are not confused in section 6.
zero_cnt_hours = int((hour["cnt"] == 0).sum())
print("Hourly rows with cnt == 0:", zero_cnt_hours)
record("Hourly rows with cnt == 0", str(zero_cnt_hours), "info")

Hourly rows with cnt == 0: 0


## 6. Missing hours against a complete calendar

The README reports 17,379 hourly records. A full calendar over 731 days would have 731 × 24 = 17,544 hours.

We build that full calendar and list every timestamp absent from `hour.csv`. **An absent row is not the same as an hour with zero rentals.** The data does not tell us why an hour is missing; candidates include zero rentals not recorded, system downtime, or a data-collection gap. For now we only describe the gaps and do not fill them.

In [13]:
full_calendar = pd.date_range(
    start=hour["date"].min(),
    end=hour["date"].max() + pd.Timedelta(hours=23),
    freq="h",
)
observed = pd.DatetimeIndex(hour["timestamp"])
missing_ts = full_calendar.difference(observed)
extra_ts = observed.difference(full_calendar)  # should be empty

print(f"Expected hourly timestamps: {len(full_calendar):,}")
print(f"Observed hourly timestamps: {observed.nunique():,}")
print(f"Absent from hour.csv:       {len(missing_ts):,}")
print(f"Outside the calendar:       {len(extra_ts)}")

record("Absent hourly timestamps",
       f"{len(missing_ts)} of {len(full_calendar):,} expected ({len(missing_ts) / len(full_calendar):.2%})",
       "issue" if len(missing_ts) else "pass")

Expected hourly timestamps: 17,544
Observed hourly timestamps: 17,379
Absent from hour.csv:       165
Outside the calendar:       0


In [14]:
missing = pd.DataFrame({"timestamp": missing_ts})
missing["date"] = missing["timestamp"].dt.normalize()
missing["hr"] = missing["timestamp"].dt.hour

print("Sample of absent timestamps:")
print(missing["timestamp"].head(10).to_string(index=False))

Sample of absent timestamps:
2011-01-02 05:00:00
2011-01-03 02:00:00
2011-01-03 03:00:00
2011-01-04 03:00:00
2011-01-05 03:00:00
2011-01-06 03:00:00
2011-01-07 03:00:00
2011-01-11 03:00:00
2011-01-11 04:00:00
2011-01-12 03:00:00


In [15]:
# Descriptive breakdown only: how the absent hours are spread across days and hours.
per_day = missing.groupby("date").size().rename("absent_hours")
print("Days with at least one absent hour:", len(per_day))
print("Days with all 24 hours present:   ", day["date"].nunique() - len(per_day))
print()
print("Distribution: number of days by count of absent hours")
print(per_day.value_counts().sort_index().rename_axis("absent_hours_in_day").rename("days").to_string())
print()
print("Days with the most absent hours:")
print(per_day.sort_values(ascending=False).head(10).to_string())

Days with at least one absent hour: 76
Days with all 24 hours present:    655

Distribution: number of days by count of absent hours
absent_hours_in_day
1     62
2      6
6      2
7      1
8      1
12     1
13     1
16     1
23     1

Days with the most absent hours:
date
2012-10-29    23
2011-01-27    16
2012-10-30    13
2011-01-18    12
2011-01-26     8
2011-08-28     7
2011-08-27     6
2011-02-22     6
2011-02-11     2
2011-01-03     2


In [16]:
print("Absent timestamps by hour of day:")
print(missing["hr"].value_counts().sort_index().rename_axis("hr").rename("absent").to_string())

Absent timestamps by hour of day:
hr
0      5
1      7
2     16
3     34
4     34
5     14
6      6
7      4
8      4
9      4
10     4
11     4
12     3
13     2
14     2
15     2
16     1
17     1
18     3
19     3
20     3
21     3
22     3
23     3


The breakdown separates two kinds of gap:
- **Isolated gaps**: days missing only 1–2 hours.
- **Long gaps**: days missing many hours in a row.

The isolated gaps cluster in the early morning, when demand is lowest. We also list which hours are missing on the days with long gaps, and on the two dates when US clocks went forward (2011-03-13 and 2012-03-11). The DST dates come from general knowledge, not from the files.

In [17]:
isolated = missing[missing["date"].map(per_day) <= 2]
print("Isolated absent hours (days with <= 2 absent):", len(isolated))
print(isolated["hr"].value_counts().sort_index().rename_axis("hr").rename("absent").to_string())
print()

print("Absent hours on days with long gaps:")
for d in per_day[per_day > 2].sort_index().index:
    print(f"  {d.date()}: {sorted(missing.loc[missing['date'] == d, 'hr'].tolist())}")
print()

for d in ["2011-03-13", "2012-03-11"]:
    print(f"Absent hours on {d} (US clocks go forward):",
          sorted(missing.loc[missing["date"] == d, "hr"].tolist()))

print()
print("Observed hours with cnt == 1:", int((hour["cnt"] == 1).sum()))

Isolated absent hours (days with <= 2 absent): 74
hr
1     1
2    10
3    27
4    27
5     8
6     1

Absent hours on days with long gaps:
  2011-01-18: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
  2011-01-26: [3, 4, 18, 19, 20, 21, 22, 23]
  2011-01-27: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  2011-02-22: [0, 1, 2, 3, 4, 5]
  2011-08-27: [18, 19, 20, 21, 22, 23]
  2011-08-28: [0, 1, 2, 3, 4, 5, 6]
  2012-10-29: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
  2012-10-30: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Absent hours on 2011-03-13 (US clocks go forward): [2]
Absent hours on 2012-03-11 (US clocks go forward): [2]

Observed hours with cnt == 1: 158


In [18]:
# Dates present in day.csv that have no hourly rows at all
days_without_hours = sorted(set(day["date"]) - set(hour["date"]))
print("day.csv dates with zero hourly rows:", [d.date() for d in days_without_hours])
record("day.csv dates with no hourly rows", str(len(days_without_hours)),
       "pass" if not days_without_hours else "issue")

day.csv dates with zero hourly rows: []


## 7. Hourly sums against daily totals

If `day.csv` is the hourly data summed by date, both files describe the same rentals and can be used together. A mismatch would mean one of them is processed differently, or that absent hours carried rentals that only the daily file includes.

In [19]:
hourly_by_day = hour.groupby("date")[COUNT_COLS].sum()
compare = day.set_index("date")[COUNT_COLS].join(hourly_by_day.add_suffix("_from_hours"), how="left")

for col in COUNT_COLS:
    compare[f"{col}_diff"] = compare[col] - compare[f"{col}_from_hours"]

diff_cols = [f"{c}_diff" for c in COUNT_COLS]
mismatch_days = compare[(compare[diff_cols] != 0).any(axis=1)]

print("Days compared:", len(compare))
print("Days where daily total != sum of hourly rows:", len(mismatch_days))
print("Total absolute difference in cnt:", int(compare["cnt_diff"].abs().sum()))
if len(mismatch_days):
    print(mismatch_days.head(10))

record("Daily totals = sum of hourly rows",
       f"{len(mismatch_days)} of {len(compare)} days differ",
       "pass" if len(mismatch_days) == 0 else "issue")

Days compared: 731
Days where daily total != sum of hourly rows: 0
Total absolute difference in cnt: 0


## Summary of checks

`status` meanings:
- `pass`: no problem found
- `issue`: something to discuss before choosing a treatment
- `info`: descriptive only

In [20]:
summary = pd.DataFrame(results)
summary

,check,result,status
0,hour.csv shape,"17,379 rows x 17 cols",info
1,day.csv shape,731 rows x 16 cols,info
2,Date parsing (%d-%m-%Y),all parsed; 2011-01-01 to 2012-12-31; month ma...,pass
3,Blank values,"hour=0, day=0",pass
4,Duplicate rows (incl. ignoring instant),"hour=0/0, day=0/0",pass
5,Duplicate hourly timestamps,0,pass
6,Duplicate daily dates,0,pass
7,Negative counts (both files),0,pass
8,Non-integer counts (both files),0,pass
9,cnt = casual + registered,"mismatches: hour=0, day=0",pass


**Not done in this notebook (on purpose):**
- No rows were filled, dropped or otherwise changed.
- No outlier or plausibility checks on the weather columns.
- No consistency check of `season`, `holiday`, `weekday` or `workingday` between the two files.
- No exploratory analysis or modelling.

These are left for the next review.